# 05 — SOP Retrieval Evaluation (Local Baseline)

**Goal:** create a reproducible, local baseline for retrieving relevant SOP passages and evaluate whether expected policy topics appear in the retrieved text.

This is not yet an LLM/RAG agent evaluation. It provides a simple benchmark that can later be reused to compare the actual retriever.

In [1]:
from pathlib import Path
import re

ROOT = Path.cwd()
if not (ROOT / "data" / "policy").exists() and (ROOT.parent / "data" / "policy").exists():
    ROOT = ROOT.parent
SOP_PATH = ROOT / "data" / "policy" / "Cold_Chain_Incident_SOP_v2.md"
if not SOP_PATH.exists():
    raise FileNotFoundError(f"SOP file not found: {SOP_PATH}")

sop_text = SOP_PATH.read_text(encoding="utf-8")
print("SOP characters:", len(sop_text))
print(sop_text[:1200])

SOP characters: 1410
# Southern California Logistics Operations
## Standard Operating Procedure (SOP): Cold-Chain & Transit Anomalies
**Version:** 2.4 | **Effective Date:** Jan 2021 | **Confidentiality:** Internal Operations Only

### 1. Temperature Control & Spoilage Prevention (Cold-Chain)
All refrigerated fleets must maintain strict IoT temperature compliance to prevent cargo spoilage.
* **Fresh Perishables:** IoT temperature must remain between 0.0°C and 4.0°C. 
* **Critical Breach:** If `IOT_TEMP_VAL_C` exceeds **4.0°C**, an immediate cold-chain breach is declared. 
* **Mitigation Protocol:** The dispatcher must immediately contact the driver to restart the auxiliary cooling unit. If ETA delay is greater than 1 hour, divert the vehicle to the nearest emergency cold-storage facility.

### 2. Route Congestion & Diversion Tactics
Port congestion heavily impacts SLA compliance. 
* **Port of Long Beach / LA:** If port congestion level (`PRT_CNG_LVL`) exceeds a severity index of **7.0**

## 1. Split SOP into paragraph-sized passages

In [2]:
passages = [p.strip() for p in re.split(r"\\n\\s*\\n", sop_text) if p.strip()]
print("Passages:", len(passages))
for i, passage in enumerate(passages[:5]):
    print(f"\\n[{i}] {passage[:350]}")

Passages: 1
\n[0] # Southern California Logistics Operations
## Standard Operating Procedure (SOP): Cold-Chain & Transit Anomalies
**Version:** 2.4 | **Effective Date:** Jan 2021 | **Confidentiality:** Internal Operations Only

### 1. Temperature Control & Spoilage Prevention (Cold-Chain)
All refrigerated fleets must maintain strict IoT temperature compliance to pre


## 2. Simple keyword retrieval baseline

In [3]:
def retrieve(query, passages=passages, top_k=3):
    terms = {t.lower() for t in re.findall(r"[a-zA-Z0-9.]+", query) if len(t) > 2}
    scored = []
    for idx, passage in enumerate(passages):
        words = set(re.findall(r"[a-zA-Z0-9.]+", passage.lower()))
        score = len(terms & words)
        scored.append((score, idx, passage))
    return sorted(scored, reverse=True)[:top_k]

queries = [
    "temperature above 4 degrees Celsius",
    "ETA delay more than one hour emergency cold storage",
    "port congestion above 7 Inland Empire Overflow Depot",
    "High Risk delay probability above 0.65 Tier 2 manager",
]
for query in queries:
    print(f"\\nQUERY: {query}")
    for score, idx, passage in retrieve(query):
        print(f"score={score} passage={idx}: {passage[:300]}")

\nQUERY: temperature above 4 degrees Celsius
score=1 passage=0: # Southern California Logistics Operations
## Standard Operating Procedure (SOP): Cold-Chain & Transit Anomalies
**Version:** 2.4 | **Effective Date:** Jan 2021 | **Confidentiality:** Internal Operations Only

### 1. Temperature Control & Spoilage Prevention (Cold-Chain)
All refrigerated fleets must
\nQUERY: ETA delay more than one hour emergency cold storage
score=7 passage=0: # Southern California Logistics Operations
## Standard Operating Procedure (SOP): Cold-Chain & Transit Anomalies
**Version:** 2.4 | **Effective Date:** Jan 2021 | **Confidentiality:** Internal Operations Only

### 1. Temperature Control & Spoilage Prevention (Cold-Chain)
All refrigerated fleets must
\nQUERY: port congestion above 7 Inland Empire Overflow Depot
score=6 passage=0: # Southern California Logistics Operations
## Standard Operating Procedure (SOP): Cold-Chain & Transit Anomalies
**Version:** 2.4 | **Effective Date:** Jan 2021 | **Confiden

## 3. Small expected-topic coverage check

In [4]:
test_cases = [
    {"query": "temperature above 4 degrees Celsius", "expected": ["4.0", "cooling"]},
    {"query": "ETA delay more than one hour emergency cold storage", "expected": ["1 hour", "cold-storage"]},
    {"query": "port congestion above 7 Inland Empire Overflow Depot", "expected": ["7.0", "Inland Empire"]},
    {"query": "High Risk delay probability above 0.65 Tier 2 manager", "expected": ["0.65", "Tier 2"]},
]

rows = []
for case in test_cases:
    results = retrieve(case["query"], top_k=3)
    retrieved_text = " ".join(item[2] for item in results).lower()
    hits = [term for term in case["expected"] if term.lower() in retrieved_text]
    rows.append({
        "query": case["query"],
        "expected_terms": ", ".join(case["expected"]),
        "matched_terms": ", ".join(hits),
        "coverage": len(hits) / len(case["expected"])
    })

import pandas as pd
evaluation = pd.DataFrame(rows)
display(evaluation)
print("Mean expected-term coverage:", round(evaluation["coverage"].mean(), 3))

,query,expected_terms,matched_terms,coverage
0,temperature above 4 degrees Celsius,"4.0, cooling","4.0, cooling",1.0
1,ETA delay more than one hour emergency cold st...,"1 hour, cold-storage","1 hour, cold-storage",1.0
2,port congestion above 7 Inland Empire Overflow...,"7.0, Inland Empire","7.0, Inland Empire",1.0
3,High Risk delay probability above 0.65 Tier 2 ...,"0.65, Tier 2","0.65, Tier 2",1.0


Mean expected-term coverage: 1.0


## Limitations and next step

This lexical baseline is intentionally simple. It does not measure semantic relevance, answer correctness, citation precision, or hallucination rate. Once the LangGraph agent and production retriever are implemented, replace this baseline comparison with tests that evaluate retrieved source passages, answer grounding, and citation correctness.